<a href="https://colab.research.google.com/github/Gopichandsimats/ITA0605-Meachine-Learning/blob/main/1%2C2%2C3%2C4%2C5%2C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# FIND-S Algorithm

import pandas as pd

# Training dataset
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Strong'],
    'Water': ['Warm', 'Warm', 'Warm', 'Warm'],
    'Forecast': ['Same', 'Same', 'Change', 'Same'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)

print("Training Dataset:")
print(df)

# FIND-S Algorithm
def find_s_algorithm(data):
    hypothesis = None

    for index, row in data.iterrows():

        if row['EnjoySport'] == 'Yes':

            instance = list(row[:-1])

            if hypothesis is None:
                hypothesis = instance.copy()

            else:
                for i in range(len(hypothesis)):
                    if hypothesis[i] != instance[i]:
                        hypothesis[i] = '?'

            print("After training example", index + 1, ":", hypothesis)

    return hypothesis

# Find the most specific hypothesis
final_hypothesis = find_s_algorithm(df)

print("\nFinal Most Specific Hypothesis:")
print(final_hypothesis)


Training Dataset:
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High  Strong  Warm   Change         No
3  Sunny    Warm     High  Strong  Warm     Same        Yes
After training example 1 : ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
After training example 2 : ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
After training example 4 : ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

Final Most Specific Hypothesis:
['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']


In [2]:

# Candidate Elimination Algorithm

import numpy as np

data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
]

def candidate_elimination(data):
    S = ['0'] * 6
    G = [['?'] * 6]

    for example in data:
        x = example[:-1]
        label = example[-1]

        if label == 'Yes':
            G = [g for g in G if all(
                g[i] == '?' or g[i] == x[i]
                for i in range(6)
            )]

            for i in range(6):
                if S[i] == '0':
                    S[i] = x[i]
                elif S[i] != x[i]:
                    S[i] = '?'

        else:
            new_G = []
            for g in G:
                if all(g[i] == '?' or g[i] == x[i]
                       for i in range(6)):
                    for i in range(6):
                        if S[i] != '?' and S[i] != '0':
                            if S[i] != x[i]:
                                new_h = g.copy()
                                new_h[i] = S[i]
                                new_G.append(new_h)
                else:
                    new_G.append(g)

            G = new_G

    return S, G

S, G = candidate_elimination(data)

print("Final Specific Boundary (S):")
print(S)

print("\nFinal General Boundary (G):")
for hypothesis in G:
    print(hypothesis)


Final Specific Boundary (S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

Final General Boundary (G):
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']


In [3]:

# ID3 Decision Tree Algorithm

import pandas as pd
import math
from collections import Counter

data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain',
                'Rain', 'Rain', 'Overcast', 'Sunny',
                'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild',
                    'Cool', 'Cool', 'Cool', 'Mild',
                    'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak',
             'Weak', 'Strong', 'Strong', 'Weak',
             'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No',
             'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes',
             'Yes', 'No']
}

df = pd.DataFrame(data)

def entropy(target):
    counts = Counter(target)
    total = len(target)
    return -sum((count / total) * math.log2(count / total)
                for count in counts.values())

def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weighted_entropy += (len(subset) / len(df)) * entropy(subset[target])

    return total_entropy - weighted_entropy

def id3(df, target, features):
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if not features:
        return df[target].mode()[0]

    best_feature = max(
        features,
        key=lambda feature: information_gain(df, feature, target)
    )

    tree = {best_feature: {}}

    for value in df[best_feature].unique():
        subset = df[df[best_feature] == value]
        remaining = [f for f in features if f != best_feature]

        if subset.empty:
            tree[best_feature][value] = df[target].mode()[0]
        else:
            tree[best_feature][value] = id3(
                subset, target, remaining
            )

    return tree

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    if value not in tree[attribute]:
        return 'Unknown'

    return predict(tree[attribute][value], sample)

features = ['Outlook', 'Temperature', 'Humidity', 'Wind']

tree = id3(df, 'Play', features)

print("Decision Tree:")
print(tree)

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = predict(tree, new_sample)

print("\nNew Sample:", new_sample)
print("Predicted Class:", result)


Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class: No


In [4]:

# Artificial Neural Network using Backpropagation

import numpy as np

# Training dataset (XOR problem)
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([[0], [1], [1], [0]])

# Initialize weights and biases
np.random.seed(42)

input_neurons = 2
hidden_neurons = 4
output_neurons = 1

W1 = np.random.randn(input_neurons, hidden_neurons)
b1 = np.zeros((1, hidden_neurons))

W2 = np.random.randn(hidden_neurons, output_neurons)
b2 = np.zeros((1, output_neurons))

# Sigmoid activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Derivative of sigmoid
def sigmoid_derivative(x):
    return x * (1 - x)

# Training using Backpropagation
learning_rate = 0.5
epochs = 10000

for epoch in range(epochs):

    # Forward propagation
    hidden_input = np.dot(X, W1) + b1
    hidden_output = sigmoid(hidden_input)

    final_input = np.dot(hidden_output, W2) + b2
    predicted_output = sigmoid(final_input)

    # Calculate error
    error = y - predicted_output

    # Backpropagation
    d_output = error * sigmoid_derivative(predicted_output)

    error_hidden = d_output.dot(W2.T)
    d_hidden = error_hidden * sigmoid_derivative(hidden_output)

    # Update weights and biases
    W2 += hidden_output.T.dot(d_output) * learning_rate
    b2 += np.sum(d_output, axis=0, keepdims=True) * learning_rate

    W1 += X.T.dot(d_hidden) * learning_rate
    b1 += np.sum(d_hidden, axis=0, keepdims=True) * learning_rate

# Test the trained network
print("ANN Predictions:")
for i in range(len(X)):
    hidden = sigmoid(np.dot(X[i:i+1], W1) + b1)
    output = sigmoid(np.dot(hidden, W2) + b2)

    print(X[i], "=>", round(output[0][0], 3),
          "Class:", int(output[0][0] >= 0.5))

print("\nActual Output:")
print(y.flatten())


ANN Predictions:
[0 0] => 0.01 Class: 0
[0 1] => 0.987 Class: 1
[1 0] => 0.984 Class: 1
[1 1] => 0.017 Class: 0

Actual Output:
[0 1 1 0]


In [5]:

# K-Nearest Neighbours Algorithm

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target

# Split dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create KNN classifier
knn = KNeighborsClassifier(n_neighbors=3)

# Train the model
knn.fit(X_train, y_train)

# Predict test data
y_pred = knn.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("K-Nearest Neighbours Algorithm")
print("Number of Neighbours:", 3)
print("Accuracy:", accuracy * 100, "%")

# Classify a new sample
new_sample = [[5.1, 3.5, 1.4, 0.2]]

prediction = knn.predict(new_sample)

print("\nNew Sample:", new_sample)
print("Predicted Flower:", iris.target_names[prediction[0]])

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred, target_names=iris.target_names
))


K-Nearest Neighbours Algorithm
Number of Neighbours: 3
Accuracy: 100.0 %

New Sample: [[5.1, 3.5, 1.4, 0.2]]
Predicted Flower: setosa

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

